# Session 05: Feature Detection & Description

**CVI4IC - Summer Semester 2026**

Detecting keypoints and computing descriptors for feature matching.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1. Harris Corner Detection

In [ ]:
# Create a checkerboard pattern
checker = np.zeros((200, 200), dtype=np.uint8)
for i in range(0, 200, 40):
    for j in range(0, 200, 40):
        if (i // 40 + j // 40) % 2 == 0:
            checker[i:i+40, j:j+40] = 255

# Harris corner detection
harris = cv2.cornerHarris(np.float32(checker), blockSize=2, ksize=3, k=0.04)

# Mark corners
checker_color = cv2.cvtColor(checker, cv2.COLOR_GRAY2BGR)
checker_color[harris > 0.01 * harris.max()] = [0, 0, 255]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(checker, cmap="gray")
axes[0].set_title("Checkerboard")
axes[1].imshow(cv2.cvtColor(checker_color, cv2.COLOR_BGR2RGB))
axes[1].set_title("Harris Corners")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. SIFT Features

In [ ]:
# Create a more complex test image
img = np.zeros((300, 300), dtype=np.uint8)
cv2.rectangle(img, (30, 30), (120, 120), 200, -1)
cv2.circle(img, (220, 80), 50, 150, -1)
cv2.ellipse(img, (150, 230), (60, 30), 45, 0, 360, 180, -1)
cv2.putText(img, "CV", (100, 180), cv2.FONT_HERSHEY_SIMPLEX, 2, 255, 3)

sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(img, None)

img_kp = cv2.drawKeypoints(img, keypoints, None,
                           flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

plt.figure(figsize=(8, 6))
plt.imshow(img_kp)
plt.title(f"SIFT: {len(keypoints)} keypoints")
plt.axis("off")
plt.show()

if descriptors is not None:
    print(f"Descriptor shape: {descriptors.shape}")

## 3. ORB Features

In [ ]:
orb = cv2.ORB_create(nfeatures=200)
keypoints_orb, descriptors_orb = orb.detectAndCompute(img, None)

img_orb = cv2.drawKeypoints(img, keypoints_orb, None, color=(0, 255, 0))

plt.figure(figsize=(8, 6))
plt.imshow(img_orb)
plt.title(f"ORB: {len(keypoints_orb)} keypoints")
plt.axis("off")
plt.show()

## 4. Feature Matching

In [ ]:
# Create a rotated version of the image
M = cv2.getRotationMatrix2D((150, 150), 30, 0.8)
img_rotated = cv2.warpAffine(img, M, (300, 300))

# Detect and match SIFT features
kp1, desc1 = sift.detectAndCompute(img, None)
kp2, desc2 = sift.detectAndCompute(img_rotated, None)

if desc1 is not None and desc2 is not None:
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    matches = bf.match(desc1, desc2)
    matches = sorted(matches, key=lambda m: m.distance)

    img_matches = cv2.drawMatches(img, kp1, img_rotated, kp2,
                                  matches[:20], None,
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    plt.figure(figsize=(14, 5))
    plt.imshow(img_matches)
    plt.title(f"Top 20 SIFT Matches ({len(matches)} total)")
    plt.axis("off")
    plt.show()
else:
    print("Not enough features found for matching")

## Exercises

1. Compare SIFT vs. ORB keypoints on the same image (count, distribution)
2. Implement Lowe's ratio test for FLANN-based matching
3. Try matching features between two photos of the same scene from different angles

In [ ]:
# Your code here
